# RAG Assignment: Baseline vs. Cross-Encoder Re-Ranking
Document: EU AI Act (Regulation (EU) 2024/1689). Both pipelines run over the same chunks.

- **Pipeline 1 (Baseline):** Bi-Encoder -> FAISS Top 10 (cosine) -> Top 3 -> LLM -> Answer 1  
- **Pipeline 2 (Re-ranking):** the same Top 10 -> Cross-Encoder -> Top 3 -> LLM (identical prompt) -> Answer 2  
- **Failure analysis:** Keyword Trap, Exceptions, Negation, Other (buried at rank 4-10)  

In [3]:
%pip install -q pypdf sentence-transformers faiss-cpu google-genai pandas requests


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, re, json, glob, time, random, getpass, hashlib
import numpy as np
import pandas as pd
import faiss
from IPython.display import display
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

# ---------------- CONFIG ----------------
PDF_PATH        = "eu_ai_act.pdf"   # EU AI Act PDF, saved from https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"   # Bi-Encoder
CROSS_MODEL     = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # Cross-Encoder
GEMINI_MODEL    = None      # None = use the best Gemini model the API key can call
CHUNK_WORDS     = 120
CHUNK_OVERLAP   = 24
TOP_K_RETRIEVE  = 10
TOP_K_FINAL     = 3
SEED            = 42
MAX_ROUNDS      = 5         # rounds of failure-query generation
MIN_PROMOTED_PER_CAT = 3    # stop generating once every category has this many retrieval failures
MAX_TRIES_PER_CAT    = 4    # per category, how many retrieval failures are tried end to end to find one where the answer really improves
MIN_OVERLAP_GAIN     = 0.25 # Answer 2 must cover at least this much more of the reference answer than Answer 1 (or Answer 1 must decline)
MIN_ANSWER_OVERLAP   = 0.40 # ...and Answer 2 itself must cover at least this share of the reference answer (so it is actually right)
# ----------------------------------------
random.seed(SEED)

/Users/erva_rishitha/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/erva_rishitha/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/erva_rishitha/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Pyt

## 1. Document
The notebook uses `eu_ai_act.pdf` from this folder. **EUR-Lex blocks scripts** (it answers with an empty HTTP 202 "bot check"), so if the file is missing, open  
`https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689` in your browser, save it as `eu_ai_act.pdf` and upload it (Colab Files panel, or the upload prompt below).

In [5]:
import requests

PDF_URLS = [
    ("EU AI Act - European Parliament final text", "https://www.europarl.europa.eu/doceo/document/TA-9-2024-0138-FNL-COR01_EN.pdf"),
    ("EU AI Act - EUR-Lex Official Journal", "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"),
    ("EU AI Act - EUR-Lex (CELEX)", "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32024R1689"),
]
MIN_WORDS = 30000   # the AI Act has ~90k words; this rejects stubs and HTML error pages

def pdf_is_usable(path, min_words=MIN_WORDS):
    try:
        with open(path, "rb") as f:
            if not f.read(5).startswith(b"%PDF"):
                return False
        n = 0
        for p in PdfReader(path).pages:
            n += len((p.extract_text() or "").split())
            if n >= min_words:
                return True
        return False
    except Exception:
        return False

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def try_download(name, url, out="document.pdf"):
    try:
        r = requests.get(url, timeout=120, headers=HEADERS, allow_redirects=True)
        if r.status_code == 200 and r.content.startswith(b"%PDF"):
            open(out, "wb").write(r.content)
            if pdf_is_usable(out):
                return out
        print(f"  - {name}: not usable (HTTP {r.status_code}, {len(r.content)} bytes)")
    except Exception as e:
        print(f"  - {name}: failed ({str(e)[:70]})")
    return None

def get_pdf():
    if PDF_PATH and os.path.exists(PDF_PATH):
        return PDF_PATH
    if PDF_PATH:
        print(f"{PDF_PATH} not found. Trying automatic download...")
    for name, url in PDF_URLS:
        p = try_download(name, url)
        if p:
            print("Downloaded:", name); return p
    for p in glob.glob("*.pdf"):
        if pdf_is_usable(p):
            return p
    try:
        from google.colab import files
        print("Automatic download failed (EUR-Lex bot check). Download the PDF in your browser from\n"
              "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689\nthen upload it here:")
        path = next(iter(files.upload()))
    except ImportError:
        path = input("Enter the path to the downloaded EU AI Act PDF: ").strip().strip('"')
    assert pdf_is_usable(path), "That file is not a text-based PDF with enough text. Re-download it in your browser."
    return path

PDF_PATH = get_pdf()
print("Using PDF:", PDF_PATH)

Using PDF: eu_ai_act.pdf


## 2. LLM (best available Gemini model)
The notebook lists the models the key can call, ranks them (Pro > Flash > Flash-Lite, newest first), and uses the first one that answers a test call. That one model is used for every LLM call. If its daily quota runs out, it asks for a new API key and continues with the same model.

In [4]:
def make_client(max_attempts=3):
    """Ask for the API key and test it right away. A rejected key is discarded, so re-running asks again."""
    global llm_client
    for _ in range(max_attempts):
        key = os.environ.get("GEMINI_API_KEY", "").strip().strip('"').strip("'")
        if not key:
            key = getpass.getpass("Paste your Gemini API key (from https://aistudio.google.com/apikey): ").strip().strip('"').strip("'")
        try:
            client = genai.Client(api_key=key)
            next(iter(client.models.list()))          # cheap call that fails immediately if the key is bad
            os.environ["GEMINI_API_KEY"] = key
            llm_client = client
            return
        except Exception as e:
            if "API key not valid" in str(e) or "API_KEY_INVALID" in str(e):
                print("The API key was rejected. Paste a key created at https://aistudio.google.com/apikey "
                      "(no spaces or quotes).")
                os.environ.pop("GEMINI_API_KEY", None)
                continue
            raise
    raise RuntimeError("No valid Gemini API key provided.")

make_client()

def new_key():
    """Daily quota used up: ask for a new key (new Google AI Studio project) and keep the same model."""
    global llm_client
    k = getpass.getpass(f"Quota of {MODEL} used up. Paste a NEW Gemini API key (Enter to stop): ").strip()
    if not k:
        raise RuntimeError("Stopped: quota exhausted. Cached answers are kept; re-run later or with a new key.")
    os.environ["GEMINI_API_KEY"] = k
    llm_client = genai.Client(api_key=k)

# Rank the models this key can call: Pro > Flash > Flash-Lite, newest version first
SKIP = ["image", "tts", "live", "audio", "embedding", "transcribe", "omni", "cyber", "robotics", "computer-use",
        "veo", "imagen", "learnlm", "vision", "exp", "customtools", "deep-research", "nano", "banana"]
def version(n):
    m = re.search(r"gemini-(\d+(?:\.\d+)?)", n)
    return float(m.group(1)) if m else 0.0
def rank_key(n):
    kind = 0 if "pro" in n else 1 if ("flash" in n and "lite" not in n) else 2 if "flash" in n else 3
    return (kind, -version(n), 1 if "preview" in n else 0, n)

listed = []
for m in llm_client.models.list():
    name = (getattr(m, "name", "") or "").replace("models/", "")
    actions = getattr(m, "supported_actions", None) or []
    if name.startswith("gemini") and (not actions or "generateContent" in actions) and not any(k in name for k in SKIP):
        listed.append(name)
queue = ([GEMINI_MODEL] if GEMINI_MODEL else []) + sorted([n for n in set(listed) if not n.endswith("-latest")], key=rank_key)[:8] \
        + sorted([n for n in set(listed) if n.endswith("-latest")], key=rank_key)

MODEL = None
for m in queue:                       # first model that really answers (e.g. Pro may have no free quota)
    try:
        r = llm_client.models.generate_content(model=m, contents="Reply with the single word OK",
                                               config=types.GenerateContentConfig(temperature=0))
        if r.text:
            MODEL = m; break
    except Exception as e:
        print(f"  - {m}: unavailable ({str(e)[:80].strip()})")
assert MODEL, "No working Gemini model found for this key."
print(">>> Using ONE model for the whole notebook:", MODEL)

# LLM call: temperature 0, cached on disk (per model + prompt), retries on rate limits
MODELS_USED = set()
CACHE_FILE = "llm_cache_v3.json"
try:
    LLM_CACHE = json.load(open(CACHE_FILE))
except Exception:
    LLM_CACHE = {}

def llm(prompt, json_mode=False, retries=6):
    key = hashlib.sha1((MODEL + ("|J|" if json_mode else "|T|") + prompt).encode()).hexdigest()
    if key in LLM_CACHE:
        MODELS_USED.add(MODEL)
        return LLM_CACHE[key]
    cfg = types.GenerateContentConfig(temperature=0, response_mime_type="application/json" if json_mode else None)
    attempt = 0
    while True:
        try:
            text = llm_client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text.strip()
            LLM_CACHE[key] = text
            MODELS_USED.add(MODEL)
            json.dump(LLM_CACHE, open(CACHE_FILE, "w"))
            return text
        except Exception as e:
            msg = str(e)
            if "404" in msg or "NOT_FOUND" in msg:
                raise RuntimeError(f"Model {MODEL} is not available. Set GEMINI_MODEL in the config cell.")
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                if "PerDay" in msg or "per day" in msg.lower():
                    new_key(); attempt = 0; continue
                attempt += 1
                if attempt > retries:
                    new_key(); attempt = 0; continue
                time.sleep(20); continue
            attempt += 1
            if attempt > retries:
                raise
            time.sleep(5 * attempt)

Paste your Gemini API key (from https://aistudio.google.com/apikey): ··········
  - gemini-3.1-pro-preview: unavailable (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu)
  - gemini-2.5-pro: unavailable (404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-)
>>> Using ONE model for the whole notebook: gemini-3.8-flash


## 3. Chunking
One chunk list (120-word windows, 24-word overlap, page number kept) is used by **both** pipelines.

In [6]:
reader = PdfReader(PDF_PATH)
words = []   # (word, page_number)
for page_no, page in enumerate(reader.pages, start=1):
    for w in (page.extract_text() or "").split():
        words.append((w, page_no))

chunks = []
step = CHUNK_WORDS - CHUNK_OVERLAP
for start in range(0, len(words), step):
    piece = words[start:start + CHUNK_WORDS]
    if not piece:
        break
    chunks.append({"id": len(chunks), "text": " ".join(w for w, _ in piece), "page": piece[0][1]})
print(f"Pages: {len(reader.pages)} | Words: {len(words)} | Chunks: {len(chunks)}")
assert len(chunks) > 15, "Too little text extracted (scanned PDF?)."

def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()
CHUNK_NORM = [norm(c["text"]) for c in chunks]    # used to locate the chunk(s) containing a piece of evidence

Pages: 144 | Words: 91421 | Chunks: 953


## 4. Pipeline 1: Bi-Encoder + FAISS (Top 10 by cosine similarity)

In [6]:
embedder = SentenceTransformer(EMBED_MODEL)

# Normalised embeddings + inner-product index  =>  inner product == cosine similarity
chunk_emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                            show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Stage 1: dense retrieval. Returns [(chunk_id, cosine_score), ...] best-first."""
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

## 5. Pipeline 2: Cross-Encoder re-ranking of the same Top 10

In [7]:
cross_encoder = CrossEncoder(CROSS_MODEL)

def rerank(query, candidates):
    """Stage 2: score each (query, chunk) pair jointly, then sort descending."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = sorted(zip([cid for cid, _ in candidates], ce_scores), key=lambda x: x[1], reverse=True)
    return [(int(cid), float(s)) for cid, s in out]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## 6. Answer generation (identical prompt for both pipelines)

In [8]:
PROMPT_TEMPLATE = """You are answering a question using ONLY the context below.
If the context does not contain the answer, say "The provided context does not contain the answer."

Context:
{context}

Question: {question}

Answer:"""

def generate_answer(query, chunk_ids):
    context = "\n\n".join(f"[Chunk {cid}, page {chunks[cid]['page']}]\n{chunks[cid]['text']}" for cid in chunk_ids)
    return llm(PROMPT_TEMPLATE.format(context=context, question=query))

## 7. Failure-mode analysis
**Step A.** For passages sampled from the document, the LLM writes one or two questions per failure category (keyword trap, exception, negation, other) with a reference answer and an evidence sentence copied from the passage. The *gold chunks* are all chunks containing that evidence (so the overlap between neighbouring chunks cannot create fake failures). A query is a **retrieval failure** if no gold chunk is in the baseline Top 3 (best gold rank 4-10) but the Cross-Encoder lifts one into the Top 3.

**Step B.** For each category, retrieval failures are tried from the largest rank gain downwards (at most `MAX_TRIES_PER_CAT`, never reusing an answer chunk already used by another example). Both pipelines answer, and the first query where **Answer 2 covers the reference answer well and clearly better than Answer 1** (or Answer 1 declines to answer) is kept. The overlap score is only a simple word-overlap check, not a judge: read the answers yourself. The category label comes from how the question was written, so check in Section 8 whether each example really shows that failure type.

In [9]:
CATEGORIES = {
    "keyword_trap": "Keyword Trap / Distractor",
    "exception":    "Exceptions & Corner Cases",
    "negation":     "Negation & Constraints",
    "other":        "Other (buried at rank 4-10)",
}

GEN_PROMPT = """Below are numbered passages from a document. For EACH passage, write questions that can be answered ONLY from that passage.
Every question must be fully self-contained and understandable WITHOUT seeing the passage: never say "the passage" or "this text", and never refer to internal numbering such as "condition (b)", "point 3", "paragraph 2" or "subparagraph"; name the actual subject instead.
Return a JSON array with one object per passage. Each object has "id" (the passage number) and these keys, each a list of up to 2 objects {{"question": ..., "answer": ..., "evidence": ...}} (use an empty list if the passage does not suit that category):
- "keyword_trap": a question answered by this passage but phrased with synonyms/paraphrases instead of the passage's own distinctive words, and containing generic words likely to appear in other, unrelated sections.
- "exception": a question whose answer is a special exception, derogation, penalty, or "except when / unless / shall not apply" condition stated in the passage.
- "negation": a question about what is NOT allowed / prohibited / a limit or restriction stated in the passage.
- "other": a question about a specific detail in the passage that a general search might rank below more generic sections.
"answer" = a 1-2 sentence reference answer taken from the passage.
"evidence" = the exact sentence or clause (8-40 words) copied word for word from the passage that contains the answer.

Passages:
{passages}"""

SIGNAL_WORDS = ["not ", "except", "unless", "only", "must", "cannot", "may not", "prohibited", "never", "at most", "at least", "limit", "shall not", "derogat"]
BAD_REF = re.compile(r"\b(paragraph|subparagraph|passage|point \d|condition \(|letter \()|\([a-z]\)|\(\d+\)", re.I)

def pick_source_chunks(exclude, n_random=6, n_signal=12):
    """Mostly passages rich in rules/exceptions, plus a few random ones."""
    eligible = [c for c in chunks if len(c["text"].split()) >= 90 and c["id"] not in exclude]
    signal = [c for c in eligible if sum(w in c["text"].lower() for w in SIGNAL_WORDS) >= 3]
    picked = random.sample(signal, min(n_signal, len(signal)))
    ids = {c["id"] for c in picked}
    picked += random.sample([c for c in eligible if c["id"] not in ids], min(n_random, len(eligible) - len(picked)))
    return picked

def parse_json(txt):
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    try:
        return json.loads(txt)
    except Exception:
        return {}

def gold_chunks_for(evidence, source_id):
    """All chunks that contain the evidence text (handles overlap between neighbouring chunks)."""
    ev = norm(evidence or "")
    w = ev.split()
    if len(w) >= 6:
        for probe in (ev, " ".join(w[:8]), " ".join(w[-8:])):
            hits = [i for i, t in enumerate(CHUNK_NORM) if probe in t]
            if source_id in hits:
                return hits
    return [source_id]

def generate_candidates(source_chunks, batch=6):
    """Ask the LLM for failure-category questions (+ reference answer + evidence) for each passage."""
    out = []
    for i in range(0, len(source_chunks), batch):
        group = source_chunks[i:i + batch]
        text = "\n\n".join(f"[Passage {j}]\n{c['text']}" for j, c in enumerate(group))
        data = parse_json(llm(GEN_PROMPT.format(passages=text), json_mode=True))
        items = data if isinstance(data, list) else (data.get("results", []) if isinstance(data, dict) else [])
        for item in items:
            if not isinstance(item, dict):
                continue
            try:
                c = group[int(item.get("id"))]
            except Exception:
                continue
            for key in CATEGORIES:
                qs = item.get(key) or []
                if isinstance(qs, dict):
                    qs = [qs]
                for q in qs if isinstance(qs, list) else []:
                    if not isinstance(q, dict):
                        continue
                    question, answer, evidence = q.get("question"), q.get("answer"), q.get("evidence")
                    if (isinstance(question, str) and len(question.split()) >= 4 and not BAD_REF.search(question)
                            and isinstance(answer, str) and answer.strip()):
                        out.append({"category": CATEGORIES[key], "query": question.strip(), "ref_answer": answer.strip(),
                                    "gold_ids": gold_chunks_for(evidence, c["id"])})
        time.sleep(1)
    return out

def best_gold_rank(ids, gold_ids):
    r = [ids.index(g) + 1 for g in gold_ids if g in ids]
    return min(r) if r else None

def evaluate_candidates(cands, seen):
    """Rank of the gold chunk in the baseline Top 10 and after re-ranking."""
    rows = []
    for cand in cands:
        if cand["query"] in seen:
            continue
        seen.add(cand["query"])
        base = baseline_retrieve(cand["query"])
        b = best_gold_rank([cid for cid, _ in base], cand["gold_ids"])
        if b is None:
            continue                                   # gold not in Top 10: re-ranking cannot help
        r = best_gold_rank([cid for cid, _ in rerank(cand["query"], base)], cand["gold_ids"])
        rows.append({**cand, "base_rank": b, "rerank_rank": r, "gain": b - r,
                     "promoted": (b > TOP_K_FINAL) and (r <= TOP_K_FINAL)})
    return rows

# ---------- Step A: find retrieval failures ----------
random.seed(SEED)
used_sources, seen_q, all_rows = set(), set(), []
for rnd in range(1, MAX_ROUNDS + 1):
    src = pick_source_chunks(used_sources)
    if not src:
        break
    used_sources |= {c["id"] for c in src}
    all_rows += evaluate_candidates(generate_candidates(src), seen_q)
    cand_df = pd.DataFrame(all_rows)
    counts = cand_df[cand_df["promoted"]]["category"].value_counts() if len(cand_df) else pd.Series(dtype=int)
    print(f"Round {rnd}: retrieval failures per category: { {c: int(counts.get(c, 0)) for c in CATEGORIES.values()} }")
    if all(counts.get(c, 0) >= MIN_PROMOTED_PER_CAT for c in CATEGORIES.values()):
        break
cand_df = pd.DataFrame(all_rows)
print(f"{len(cand_df)} queries had the answer chunk inside the baseline Top 10; "
      f"{int(cand_df['promoted'].sum()) if len(cand_df) else 0} are retrieval failures (rank 4-10 -> Top 3).")
if len(cand_df):
    # context for the report: how often does re-ranking help vs hurt over ALL evaluated queries?
    t3_before, t3_after = cand_df["base_rank"] <= TOP_K_FINAL, cand_df["rerank_rank"] <= TOP_K_FINAL
    effect_counts = {"evaluated": len(cand_df),
                     "moved into Top 3 (fixed)": int((~t3_before & t3_after).sum()),
                     "pushed out of Top 3 (made worse)": int((t3_before & ~t3_after).sum()),
                     "in Top 3 before and after": int((t3_before & t3_after).sum()),
                     "outside Top 3 before and after": int((~t3_before & ~t3_after).sum())}
    print("Effect of re-ranking on the answer chunk (all evaluated queries):")
    print(f"  moved INTO the Top 3 (fixed):          {int((~t3_before & t3_after).sum())}")
    print(f"  pushed OUT of the Top 3 (made worse):  {int((t3_before & ~t3_after).sum())}")
    print(f"  in the Top 3 before and after:         {int((t3_before & t3_after).sum())}")
    print(f"  outside the Top 3 before and after:    {int((~t3_before & ~t3_after).sum())}")
    display(cand_df[cand_df["promoted"]][["category", "query", "base_rank", "rerank_rank", "gain"]]
            .sort_values(["category", "gain"], ascending=[True, False]).reset_index(drop=True))

Round 1: retrieval failures per category: {'Keyword Trap / Distractor': 1, 'Exceptions & Corner Cases': 2, 'Negation & Constraints': 0, 'Other (buried at rank 4-10)': 3}
Round 2: retrieval failures per category: {'Keyword Trap / Distractor': 5, 'Exceptions & Corner Cases': 4, 'Negation & Constraints': 1, 'Other (buried at rank 4-10)': 3}
Round 3: retrieval failures per category: {'Keyword Trap / Distractor': 7, 'Exceptions & Corner Cases': 7, 'Negation & Constraints': 1, 'Other (buried at rank 4-10)': 4}
Round 4: retrieval failures per category: {'Keyword Trap / Distractor': 7, 'Exceptions & Corner Cases': 7, 'Negation & Constraints': 2, 'Other (buried at rank 4-10)': 5}
Round 5: retrieval failures per category: {'Keyword Trap / Distractor': 9, 'Exceptions & Corner Cases': 7, 'Negation & Constraints': 2, 'Other (buried at rank 4-10)': 6}
121 queries had the answer chunk inside the baseline Top 10; 24 are retrieval failures (rank 4-10 -> Top 3).
Effect of re-ranking on the answer chunk 

,category,query,base_rank,rerank_rank,gain
0,Exceptions & Corner Cases,What is one condition that allows an Annex III...,7,1,6
1,Exceptions & Corner Cases,When is an initial provider exempted from the ...,6,1,5
2,Exceptions & Corner Cases,When does the exemption for open-source genera...,7,2,5
3,Exceptions & Corner Cases,What must a conformity assessment body submit ...,5,1,4
4,Exceptions & Corner Cases,Under what circumstances is separate human ver...,5,1,4
5,Exceptions & Corner Cases,Which specific provision is excluded from the ...,5,1,4
6,Exceptions & Corner Cases,When are biometric categorisation systems excl...,4,2,2
7,Keyword Trap / Distractor,Under what framework can national governments ...,9,3,6
8,Keyword Trap / Distractor,When personal information is handled during an...,7,1,6
9,Keyword Trap / Distractor,What action may the AI Office take before disp...,5,1,4


In [10]:
# ---------- Step B: one illustrative failure per category, run through both pipelines ----------
STOP = set("this that with from have been which what when where does into than then them they their there these those under such "
           "shall must will would also only other more most some any each both over upon about between within without".split())

def content_words(text):
    return {w.rstrip("s") for w in re.findall(r"[a-z]{4,}", text.lower()) if w not in STOP}

def ref_overlap(answer, ref):
    """Share of the reference answer's content words that appear in the answer (simple word overlap, not a judge)."""
    r = content_words(ref)
    return len(r & content_words(answer)) / len(r) if r else 0.0

def declined(ans):
    return "does not contain the answer" in ans.lower()

selected, used_gold = [], set()
for cat in CATEGORIES.values():
    pool = cand_df[(cand_df["category"] == cat) & (cand_df["promoted"])].sort_values("gain", ascending=False).to_dict("records") if len(cand_df) else []
    pool = [c for c in pool if not (set(c["gold_ids"]) & used_gold)]          # a different answer chunk for every example
    if not pool:
        print(f"  [{cat}] no retrieval failure found: raise MAX_ROUNDS or change SEED and re-run.")
        continue
    tried = []
    for cand in pool[:MAX_TRIES_PER_CAT]:
        q = cand["query"]
        base = baseline_retrieve(q)
        rr = rerank(q, base)                                                   # re-ranks the SAME Top 10
        a1 = generate_answer(q, [c for c, _ in base[:TOP_K_FINAL]])            # Answer 1 (baseline Top 3)
        a2 = generate_answer(q, [c for c, _ in rr[:TOP_K_FINAL]])              # Answer 2 (re-ranked Top 3)
        o1, o2 = ref_overlap(a1, cand["ref_answer"]), ref_overlap(a2, cand["ref_answer"])
        rec = {**cand, "ans1": a1, "ans2": a2, "overlap1": o1, "overlap2": o2,
               "a1_declined": declined(a1), "a2_declined": declined(a2)}
        rec["answer_improved"] = (not rec["a2_declined"]) and o2 >= MIN_ANSWER_OVERLAP and (rec["a1_declined"] or (o2 - o1) >= MIN_OVERLAP_GAIN)
        tried.append(rec)
        print(f"  [{cat}] try {len(tried)}: chunk rank {cand['base_rank']} -> {cand['rerank_rank']} | "
              f"reference overlap A1 {o1:.0%} -> A2 {o2:.0%}{' (A1 declined)' if rec['a1_declined'] else ''}"
              f"{'  => kept' if rec['answer_improved'] else ''}")
        if rec["answer_improved"]:
            break
    best = next((t for t in tried if t["answer_improved"]), None)
    if best is None:                                                           # nothing clearly better: keep the closest one, say so
        best = max(tried, key=lambda t: t["overlap2"] - t["overlap1"])
        print(f"  [{cat}] WARNING: no tried query shows a clear answer improvement; keeping the best of {len(tried)} (rank shift only).")
    selected.append(best)
    used_gold |= set(best["gold_ids"])
print(f"\nSelected {len(selected)} failure queries.")


  [Keyword Trap / Distractor] try 1: chunk rank 9 -> 3 | reference overlap A1 60% -> A2 88%  => kept
  [Exceptions & Corner Cases] try 1: chunk rank 7 -> 1 | reference overlap A1 0% -> A2 65% (A1 declined)  => kept
  [Negation & Constraints] try 1: chunk rank 7 -> 2 | reference overlap A1 100% -> A2 100%
Quota of gemini-3.8-flash used up. Paste a NEW Gemini API key (Enter to stop): ··········
  [Negation & Constraints] try 2: chunk rank 5 -> 2 | reference overlap A1 44% -> A2 0%
  [Negation & Constraints] WARNING: no tried query shows a clear answer improvement; keeping the best of 2 (rank shift only).
  [Other (buried at rank 4-10)] try 1: chunk rank 8 -> 1 | reference overlap A1 0% -> A2 92% (A1 declined)  => kept

Selected 4 failure queries.


## 8. Both pipelines on the failure queries
For each query: rank-shift table of the Top 10 (`has_gold` marks the chunk holding the true answer), the chunks that entered the Top 3 through re-ranking, and **Answer 1 vs Answer 2** next to the reference answer. Read the chunks and answers yourself to decide whether Answer 1 omitted a clause, misstated a rule or hallucinated.

In [11]:
def show_query(rec):
    q = rec["query"]
    base = baseline_retrieve(q)
    rr = rerank(q, base)
    base_top3 = [c for c, _ in base[:TOP_K_FINAL]]
    rr_top3 = [c for c, _ in rr[:TOP_K_FINAL]]
    base_rank = {c: i + 1 for i, (c, _) in enumerate(base)}
    rr_rank = {c: i + 1 for i, (c, _) in enumerate(rr)}
    cos, ce = dict(base), dict(rr)
    table = pd.DataFrame([{
        "chunk_id": c, "page": chunks[c]["page"], "baseline_rank": base_rank[c], "reranked_rank": rr_rank[c],
        "shift": base_rank[c] - rr_rank[c],            # positive = moved up
        "cosine": round(cos[c], 3), "cross_enc": round(ce[c], 3),
        "in_base_top3": c in base_top3, "in_rerank_top3": c in rr_top3, "has_gold": c in rec["gold_ids"],
    } for c, _ in base]).sort_values("reranked_rank").reset_index(drop=True)

    print("=" * 100)
    print(f"[{rec['category']}]")
    print("QUERY:", q)
    display(table)
    print("Baseline Top 3 :", base_top3)
    print("Re-ranked Top 3:", rr_top3)
    for c in [c for c in rr_top3 if c not in base_top3]:
        print(f"\n[Chunk {c} | page {chunks[c]['page']}] entered Top 3 via re-ranking "
              f"(Rank #{base_rank[c]} -> #{rr_rank[c]}){'  <-- HOLDS THE ANSWER' if c in rec['gold_ids'] else ''}:\n{chunks[c]['text']}")
    print("\n--- Reference answer ---\n" + rec["ref_answer"])
    print(f"\n--- Answer 1 (baseline) [{MODEL}] ---\n" + rec["ans1"])
    print(f"\n--- Answer 2 (re-ranked) [{MODEL}] ---\n" + rec["ans2"])
    print(f"\nReference-answer word overlap: Answer 1 {rec['overlap1']:.0%} | Answer 2 {rec['overlap2']:.0%} "
          f"| Answer 1 declined: {rec['a1_declined']} | clear answer improvement: {rec['answer_improved']}")

for rec in selected:
    show_query(rec)

[Keyword Trap / Distractor]
QUERY: Under what framework can national governments permit law enforcement to deploy live remote biometric scanning in areas open to the public?


,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,223,25,5,1,4,0.712,3.419,False,True,False
1,95,11,8,2,6,0.700,3.297,False,True,False
2,440,53,9,3,6,0.698,2.574,False,True,True
3,442,53,1,4,-3,0.741,2.393,True,False,False
4,85,10,4,5,-1,0.724,2.305,False,False,False
5,435,52,2,6,-4,0.731,2.090,True,False,False
6,92,11,6,7,-1,0.708,1.715,False,False,False
7,86,10,3,8,-5,0.728,1.273,True,False,False
8,91,11,10,9,1,0.698,-0.421,False,False,False
9,431,52,7,10,-3,0.707,-0.886,False,False,False


Baseline Top 3 : [442, 435, 86]
Re-ranked Top 3: [223, 95, 440]

[Chunk 223 | page 25] entered Top 3 via re-ranking (Rank #5 -> #1):
not be used in the framework of law enforcement to lead to indiscriminate surveillance. The conditions for post-remote biometric identification should in any case not provide a basis to circumvent the conditions of the prohibition and strict exceptions for real time remote biometric identification. (96) In order to efficiently ensure that fundamental rights are protected, deployers of high-risk AI systems that are bodies governed by public law, or private entities providing public services and deployers of certain high-risk AI systems listed in an annex to this Regulation, such as banking or insurance entities, should carry out a fundamental rights impact assessment prior to putting it into use. Services important for individuals that are of public nature may also be provided

[Chunk 95 | page 11] entered Top 3 via re-ranking (Rank #8 -> #2):
the specific

,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,446,53,7,1,6,0.664,6.282,False,True,True
1,445,53,1,2,-1,0.707,4.786,True,True,False
2,451,54,4,3,1,0.669,4.347,False,True,False
3,620,81,9,4,5,0.658,3.692,False,False,False
4,619,81,2,5,-3,0.697,3.153,True,False,False
5,596,78,8,6,2,0.662,2.975,False,False,False
6,456,55,6,7,-1,0.665,2.773,False,False,False
7,597,78,3,8,-5,0.677,2.761,True,False,False
8,741,101,5,9,-4,0.666,1.089,False,False,False
9,207,23,10,10,0,0.654,0.195,False,False,False


Baseline Top 3 : [445, 619, 597]
Re-ranked Top 3: [446, 445, 451]

[Chunk 446 | page 53] entered Top 3 via re-ranking (Rank #7 -> #1)  <-- HOLDS THE ANSWER:
high-risk. OJ L, 12.7.2024 EN ELI: http://data.europa.eu/eli/reg/2024/1689/oj 53/144 3. By derogation from paragraph 2, an AI system referred to in Annex III shall not be considered to be high-risk where it does not pose a significant risk of harm to the health, safety or fundamental rights of natural persons, including by not materially influencing the outcome of decision making. The first subparagraph shall apply where any of the following conditions is fulfilled: (a) the AI system is intended to perform a narrow procedural task; (b) the AI system is intended to improve the result of a previously completed human activity; (c) the AI system is intended to detect decision-making patterns or deviations from prior decision-making patterns and is not meant

[Chunk 451 | page 54] entered Top 3 via re-ranking (Rank #4 -> #3):
provided f

,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,392,45,3,1,2,0.726,6.235,True,True,False
1,391,45,7,2,5,0.660,5.596,False,True,True
2,57,7,2,3,-1,0.771,5.564,True,True,False
3,56,7,1,4,-3,0.782,4.668,True,False,False
4,54,7,9,5,4,0.656,4.115,False,False,False
5,55,7,6,6,0,0.669,3.129,False,False,False
6,396,46,4,7,-3,0.702,1.486,False,False,False
7,26,4,10,8,2,0.650,0.660,False,False,False
8,121,14,5,9,-4,0.679,-0.142,False,False,False
9,397,46,8,10,-2,0.657,-2.561,False,False,False


Baseline Top 3 : [56, 57, 392]
Re-ranked Top 3: [392, 391, 57]

[Chunk 391 | page 45] entered Top 3 via re-ranking (Rank #7 -> #2)  <-- HOLDS THE ANSWER:
Article 57 applies only in so far as the requirements for high-risk AI systems under this Regulation have been integrated in that Union harmonisation legislation. 3. This Regulation does not apply to areas outside the scope of Union law, and shall not, in any event, affect the competences of the Member States concerning national security, regardless of the type of entity entrusted by the Member States with carrying out tasks in relation to those competences. This Regulation does not apply to AI systems where and in so far they are placed on the market, put into service, or used with or without modification exclusively for military, defence or national security purposes, regardless of the type of entity carrying out those

--- Reference answer ---
It does not apply to AI systems placed on the market, put into service, or used exclusive

,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,229,26,8,1,7,0.599,3.977,False,True,True
1,231,26,9,2,7,0.590,3.215,False,True,False
2,232,26,3,3,0,0.624,1.981,True,True,False
3,260,29,7,4,3,0.603,1.404,False,False,False
4,234,26,6,5,1,0.603,0.766,False,False,False
5,647,86,5,6,-1,0.610,-0.280,False,False,False
6,261,30,10,7,3,0.589,-0.925,False,False,False
7,947,142,1,8,-7,0.649,-1.034,True,False,False
8,230,26,4,9,-5,0.617,-1.106,False,False,False
9,948,143,2,10,-8,0.635,-1.527,True,False,False


Baseline Top 3 : [947, 948, 232]
Re-ranked Top 3: [229, 231, 232]

[Chunk 229 | page 26] entered Top 3 via re-ranking (Rank #8 -> #1)  <-- HOLDS THE ANSWER:
(97) The notion of general-purpose AI models should be clearly defined and set apart from the notion of AI systems to enable legal certainty. The definition should be based on the key functional characteristics of a general-purpose AI model, in particular the generality and the capability to competently perform a wide range of distinct tasks. These models are typically trained on large amounts of data, through various methods, such as self-supervised, unsupervised or reinforcement learning. General-purpose AI models may be placed on the market in various ways, including through libraries, application programming interfaces (APIs), as direct download, or as physical copy. These models may be further modified or fine-tuned into new models. Although AI models are essential components of AI

[Chunk 231 | page 26] entered Top 3 via re-r

### Evidence tables: rank shifts and answer comparison

In [12]:
rank_tbl = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "gold_chunk(s)": ",".join(map(str, r["gold_ids"])),
    "baseline_rank": r["base_rank"], "reranked_rank": r["rerank_rank"],
    "rank_change": f"Rank #{r['base_rank']} -> #{r['rerank_rank']}",
} for r in selected])
print("TABLE 1 - Rank shifts of the chunk holding the answer")
display(rank_tbl)

ans_tbl = pd.DataFrame([{
    "category": r["category"], "query": r["query"][:80],
    "answer1 (baseline)": r["ans1"][:300], "answer2 (re-ranked)": r["ans2"][:300],
    "A1 declined": r["a1_declined"], "overlap A1": f"{r['overlap1']:.0%}", "overlap A2": f"{r['overlap2']:.0%}",
} for r in selected])
print("\nTABLE 2 - Answer 1 (baseline) vs Answer 2 (re-ranked)")
display(ans_tbl)


TABLE 1 - Rank shifts of the chunk holding the answer


,category,query,gold_chunk(s),baseline_rank,reranked_rank,rank_change
0,Keyword Trap / Distractor,Under what framework can national governments ...,440,9,3,Rank #9 -> #3
1,Exceptions & Corner Cases,What is one condition that allows an Annex III...,446,7,1,Rank #7 -> #1
2,Negation & Constraints,To what types of AI systems does the Regulatio...,391,7,2,Rank #7 -> #2
3,Other (buried at rank 4-10),What are the common distribution channels thro...,229,8,1,Rank #8 -> #1



TABLE 2 - Answer 1 (baseline) vs Answer 2 (re-ranked)


,category,query,answer1 (baseline),answer2 (re-ranked),A1 declined,overlap A1,overlap A2
0,Keyword Trap / Distractor,Under what framework can national governments ...,"Based on the provided context, national govern...","Based on the provided context, a Member State ...",False,60%,88%
1,Exceptions & Corner Cases,What is one condition that allows an Annex III...,The provided context does not contain the answer.,"Based on the provided context, one condition t...",True,0%,65%
2,Negation & Constraints,To what types of AI systems does the Regulatio...,"Based on the provided context, the Regulation ...","Based on the provided context, the Regulation ...",False,100%,100%
3,Other (buried at rank 4-10),What are the common distribution channels thro...,The provided context does not contain the answer.,"Based on the provided context, general-purpose...",True,0%,92%


### Why did the baseline pick the wrong chunks? (distractor check)
For every selected failure query this prints the baseline Top-3 chunks with their cosine score and Cross-Encoder score.
Read them and write down in the report (a) what words they share with the query (keyword trap) and (b) what they say instead of the answer.


In [13]:
def show_distractors(rec, n_chars=420):
    q = rec["query"]
    base = baseline_retrieve(q)
    rr = rerank(q, base)
    ce = dict(rr)
    q_words = content_words(q)
    print("=" * 100)
    print(f"[{rec['category']}] {q}")
    for rank, (cid, cos) in enumerate(base[:TOP_K_FINAL], 1):
        shared = sorted(q_words & content_words(chunks[cid]["text"]))
        print(f"\n  Baseline #{rank} | chunk {cid} | page {chunks[cid]['page']} | cosine {cos:.3f} | cross-enc {ce[cid]:.2f}"
              f"{'  <-- HOLDS THE ANSWER' if cid in rec['gold_ids'] else ''}")
        print(f"  words shared with the query: {shared}")
        print("  " + chunks[cid]["text"][:n_chars].replace("\n", " ") + " ...")

for rec in selected:
    show_distractors(rec)


[Keyword Trap / Distractor] Under what framework can national governments permit law enforcement to deploy live remote biometric scanning in areas open to the public?

  Baseline #1 | chunk 442 | page 53 | cosine 0.741 | cross-enc 2.39
  words shared with the query: ['biometric', 'enforcement', 'national', 'remote']
  following the adoption thereof. Member States may introduce, in accordance with Union law, more restrictive laws on the use of remote biometric identification systems. 6. National market surveillance authorities and the national data protection authorities of Member States that have been notified of the use of ‘real-time’ remote biometric identification systems in publicly accessible spaces for law enforcement purpos ...

  Baseline #2 | chunk 435 | page 52 | cosine 0.731 | cross-enc 2.09
  words shared with the query: ['biometric', 'enforcement', 'national', 'remote']
  and scale of those consequences. In addition, the use of ‘real-time’ remote biometric identification s

## 8b. Hand-written queries (your own failure cases)
The failure queries above were generated by the LLM. Here you write your own, which shows you understand the failure modes.
`must_contain` is a list of lowercase phrases copied from the PDF text: every chunk containing **all** of them is treated as the gold chunk.
Use `find_chunks("some phrase")` first to check the phrase really exists in the extracted text (PDF extraction can change spacing).
Keep the queries where the gold chunk is at baseline rank 4-10 and the Cross-Encoder lifts it into the Top 3; if a query already works in the baseline, it is simply not a failure case.


In [14]:
def find_chunks(phrase):
    """Which chunk ids contain this phrase? (use it to pick a safe must_contain phrase)"""
    p = norm(phrase)
    hits = [i for i, t in enumerate(CHUNK_NORM) if p in t]
    print(f"{len(hits)} chunk(s) contain '{phrase}': {hits[:15]}")
    return hits

def run_manual(query, must_contain, ref_answer="", category="Manual query"):
    gold = [i for i, t in enumerate(CHUNK_NORM) if all(norm(p) in t for p in must_contain)]
    if not gold:
        print("No chunk contains all must_contain phrases -> adjust them with find_chunks() first."); return None
    base = baseline_retrieve(query)
    rr = rerank(query, base)
    b = best_gold_rank([c for c, _ in base], gold)
    r = best_gold_rank([c for c, _ in rr], gold)
    if b is None:
        print(f"Gold chunk(s) {gold} are NOT in the baseline Top 10 -> re-ranking cannot help (first-stage recall failure)."); return None
    a1 = generate_answer(query, [c for c, _ in base[:TOP_K_FINAL]])
    a2 = generate_answer(query, [c for c, _ in rr[:TOP_K_FINAL]])
    rec = {"category": category, "query": query, "ref_answer": ref_answer or "(none written)", "gold_ids": gold,
           "base_rank": b, "rerank_rank": r, "gain": b - r,
           "ans1": a1, "ans2": a2,
           "overlap1": ref_overlap(a1, ref_answer) if ref_answer else 0.0,
           "overlap2": ref_overlap(a2, ref_answer) if ref_answer else 0.0,
           "a1_declined": declined(a1), "a2_declined": declined(a2), "answer_improved": None}
    print(f"Gold chunk rank: baseline #{b} -> re-ranked #{r}   (retrieval failure fixed: {b > TOP_K_FINAL and r <= TOP_K_FINAL})")
    show_query(rec)
    return rec

# Candidate queries about the EU AI Act. Check the phrases with find_chunks() and change them if needed.
manual_specs = [
    dict(category="Negation & Constraints",
         query="Can an employer use AI to detect the emotions of its staff at work?",
         must_contain=["infer emotions of a natural person in the areas of workplace"],
         ref_answer="No. AI systems that infer emotions of people in the workplace and in education institutions are prohibited, except for medical or safety reasons."),
    dict(category="Negation & Constraints",
         query="Are police allowed to use live facial recognition in public places?",
         must_contain=["remote biometric identification systems in publicly accessible spaces for the purposes of law enforcement"],
         ref_answer="Real-time remote biometric identification in publicly accessible spaces for law enforcement is prohibited unless strictly necessary for specific listed objectives."),
    dict(category="Exceptions & Corner Cases",
         query="What is the maximum fine for using a prohibited AI practice?",
         must_contain=["administrative fines of up to eur 35 000 000"],
         ref_answer="Up to EUR 35 000 000 or, for an undertaking, up to 7 % of total worldwide annual turnover, whichever is higher."),
]
manual_results = [run_manual(**s) for s in manual_specs]


Gold chunk(s) [430] are NOT in the baseline Top 10 -> re-ranking cannot help (first-stage recall failure).
Gold chunk rank: baseline #8 -> re-ranked #4   (retrieval failure fixed: False)
[Negation & Constraints]
QUERY: Are police allowed to use live facial recognition in public places?


,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,86,10,7,1,6,0.532,-3.105,False,True,False
1,85,10,2,2,0,0.546,-3.858,True,True,False
2,103,12,1,3,-2,0.550,-3.898,True,True,False
3,435,52,10,4,6,0.513,-4.386,False,False,True
4,430,51,4,5,-1,0.541,-6.093,False,False,False
5,431,52,8,6,2,0.530,-7.013,False,False,True
6,429,51,3,7,-4,0.541,-7.510,True,False,False
7,91,11,6,8,-2,0.535,-8.419,False,False,False
8,222,25,5,9,-4,0.536,-8.702,False,False,False
9,434,52,9,10,-1,0.522,-9.238,False,False,False


Baseline Top 3 : [103, 85, 429]
Re-ranked Top 3: [86, 85, 103]

[Chunk 86 | page 10] entered Top 3 via re-ranking (Rank #7 -> #1):
regarding the threats, the victims or perpetrator. The use of the real-time remote biometric identification system in publicly accessible spaces should be authorised only if the relevant law enforcement authority has completed a fundamental rights impact assessment and, unless provided otherwise in this Regulation, has registered the system in the database as set out in this Regulation. The reference database of persons should be appropriate for each use case in each of the situations mentioned above. (35) Each use of a ‘real-time’ remote biometric identification system in publicly accessible spaces for the purpose of law enforcement should be subject to an express and specific authorisation by a judicial authority or by an independent administrative authority of a Member State whose

--- Reference answer ---
Real-time remote biometric identification in pub

,chunk_id,page,baseline_rank,reranked_rank,shift,cosine,cross_enc,in_base_top3,in_rerank_top3,has_gold
0,828,115,2,1,1,0.644,5.979,True,True,True
1,838,117,5,2,3,0.589,3.597,False,True,False
2,425,51,1,3,-2,0.681,1.921,True,True,False
3,72,9,3,4,-1,0.614,0.958,True,False,False
4,105,12,4,5,-1,0.589,0.714,False,False,False
5,384,44,7,6,1,0.587,-2.255,False,False,False
6,147,17,10,7,3,0.580,-5.215,False,False,False
7,395,46,9,8,1,0.583,-5.292,False,False,False
8,304,34,6,9,-3,0.587,-5.419,False,False,False
9,313,35,8,10,-2,0.584,-8.560,False,False,False


Baseline Top 3 : [425, 828, 72]
Re-ranked Top 3: [828, 838, 425]

[Chunk 838 | page 117] entered Top 3 via re-ranking (Rank #5 -> #2):
concerned with regard to the same subject matter; (e) any similar previous infringements by the Union institution, body, office or agency; (f) the manner in which the infringement became known to the European Data Protection Supervisor, in particular whether, and if so to what extent, the Union institution, body, office or agency notified the infringement; (g) the annual budget of the Union institution, body, office or agency. 2. Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 1 500 000. 3. The non-compliance of the AI system with any requirements or obligations under this Regulation, other than those laid down in Article 5, shall be subject

--- Reference answer ---
Up to EUR 35 000 000 or, for an undertaking, up to 7 % of total worldwide annual turnover, whichever i

### 8c. Quick scan for a real negation / exception failure (no LLM calls, so no quota used)
In the run above, the Negation example (Art. 2(3)) moved up in rank but Answer 1 was already correct, so it is **not** a real failure.
This cell tests several prohibition / "does not apply" questions on retrieval only. Look for rows with `base_rank` 4-10 and `rerank_rank` <= 3 and `fixed = True`.
Then run the best one with `run_manual(...)` (cell above) to get Answer 1 vs Answer 2, and use it as the Negation example in the report.
Phrases are lowercase text copied from the Act; if a row shows `gold = []`, fix the phrase with `find_chunks()`.


In [10]:
import os, re
import numpy as np
import pandas as pd
import faiss
from IPython.display import display
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Load document if not already loaded
PDF_PATH = "eu_ai_act.pdf"
if 'chunks' not in globals():
    reader = PdfReader(PDF_PATH)
    words = []
    for pno, p in enumerate(reader.pages, start=1):
        for w in (p.extract_text() or "").split():
            words.append((w, pno))
    step = 120 - 24
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + 120]
        if not piece: break
        chunks.append({"id": len(chunks), "text": " ".join(w for w, _ in piece), "page": piece[0][1]})

def norm(t):
    return re.sub(r"[^a-z0-9]+", " ", t.lower()).strip()

CHUNK_NORM = [norm(c["text"]) for c in chunks]

# 2. Build or get Bi-Encoder
if 'baseline_retrieve' not in globals():
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    chunk_emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(chunk_emb.shape[1])
    index.add(chunk_emb)
    def baseline_retrieve(query, k=10):
        q = embedder.encode([query], normalize_embeddings=True).astype("float32")
        scores, ids = index.search(q, k)
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

# 3. Build or get Cross-Encoder
if 'rerank' not in globals():
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    def rerank(query, candidates):
        pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
        ce_scores = cross_encoder.predict(pairs)
        out = sorted(zip([cid for cid, _ in candidates], ce_scores), key=lambda x: x[1], reverse=True)
        return [(int(cid), float(s)) for cid, s in out]

def best_gold_rank(ordered_ids, gold_ids):
    for r, cid in enumerate(ordered_ids, 1):
        if cid in gold_ids:
            return r
    return None

scan_specs = [
    ("Can a public authority rate citizens based on their social behaviour?",
     ["over a certain period of time based on their social behaviour"]),
    ("Is it allowed to build a face recognition database by collecting photos from the internet?",
     ["untargeted scraping of facial images from the internet or cctv footage"]),
    ("Can police predict whether a person will commit a crime just from their personality profile?",
     ["committing a criminal offence based solely on the profiling"]),
    ("Can AI be used to infer a person's race or religion from their face?",
     ["to deduce or infer their race"]),
    ("Is it allowed to influence people with techniques they are not even aware of?",
     ["subliminal techniques beyond a person s consciousness"]),
    ("Does the Act apply to AI built only for scientific research?",
     ["for the sole purpose of scientific research and development"]),
    ("Does the Act apply to free and open-source AI components?",
     ["released under free and open source licences"]),
    ("Are private individuals using AI for personal hobbies covered by the Act?",
     ["purely personal non professional activity"]),
]

def scan_retrieval(specs):
    rows = []
    for query, must in specs:
        gold = [i for i, t in enumerate(CHUNK_NORM) if all(norm(p) in t for p in must)]
        base = baseline_retrieve(query)
        rr = rerank(query, base)
        b = best_gold_rank([c for c, _ in base], gold) if gold else None
        r = best_gold_rank([c for c, _ in rr], gold) if gold else None
        rows.append({
            "query": query[:60], 
            "gold": gold[:2], 
            "base_rank": b, 
            "rerank_rank": r,
            "fixed": bool(b and r and b > 3 and r <= 3)
        })
    return pd.DataFrame(rows)

scan_df = scan_retrieval(scan_specs)
display(scan_df)


,query,gold,base_rank,rerank_rank,fixed
0,Can a public authority rate citizens based on ...,[427],9.0,5.0,False
1,Is it allowed to build a face recognition data...,"[102, 103]",1.0,1.0,False
2,Can police predict whether a person will commi...,[429],2.0,1.0,False
3,Can AI be used to infer a person's race or rel...,[431],NaN,NaN,False
4,Is it allowed to influence people with techniq...,[425],2.0,5.0,False
5,Does the Act apply to AI built only for scient...,"[58, 60]",2.0,1.0,False
6,Does the Act apply to free and open-source AI ...,"[237, 396]",4.0,5.0,False
7,Are private individuals using AI for personal ...,[395],5.0,1.0,True


## 9. Latency profiling
Retrieval only (the LLM call is identical in both pipelines and is excluded).  
- **Baseline** = embed query + FAISS search (Top 10)  
- **Re-ranking** = Cross-Encoder scoring of the 10 (query, chunk) pairs + sort  
- **Pipeline 2 total** = baseline + re-ranking

In [15]:
N_RUNS = 20
latency_queries = [r["query"] for r in selected]

# warm-up so one-off start-up cost is not measured
_w = baseline_retrieve(latency_queries[0]); rerank(latency_queries[0], _w)

rows = []
for q in latency_queries:
    b_t, r_t = [], []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        cands = baseline_retrieve(q)          # embed query + FAISS Top 10
        t1 = time.perf_counter()
        rerank(q, cands)                      # Cross-Encoder on the 10 pairs + sort
        t2 = time.perf_counter()
        b_t.append((t1 - t0) * 1000); r_t.append((t2 - t1) * 1000)
    rows.append({"query": q[:55], "baseline_ms": np.mean(b_t), "rerank_ms": np.mean(r_t),
                 "pipeline2_total_ms": np.mean(b_t) + np.mean(r_t)})

lat = pd.DataFrame(rows)
avg = {"query": "AVERAGE", **{c: lat[c].mean() for c in lat.columns if c != "query"}}
lat = pd.concat([lat, pd.DataFrame([avg])], ignore_index=True)
lat["slowdown_x"] = lat["pipeline2_total_ms"] / lat["baseline_ms"]
print(f"TABLE 3 - Latency per failure query (mean of {N_RUNS} runs, ms) | device: {getattr(cross_encoder.model, 'device', 'cpu')} | chunks indexed: {len(chunks)}")
display(lat.round(2))


TABLE 3 - Latency per failure query (mean of 20 runs, ms) | device: cpu | chunks indexed: 953


,query,baseline_ms,rerank_ms,pipeline2_total_ms,slowdown_x
0,Under what framework can national governments ...,27.76,1047.91,1075.67,38.75
1,What is one condition that allows an Annex III...,26.00,1079.35,1105.35,42.51
2,To what types of AI systems does the Regulatio...,27.22,979.43,1006.65,36.98
3,What are the common distribution channels thro...,24.32,1141.74,1166.06,47.94
4,AVERAGE,26.33,1062.11,1088.43,41.34


## 10. Report outputs

In [16]:
import textwrap

def wrap(t, indent="    "):
    return "\n".join(textwrap.fill(line, 100, initial_indent=indent, subsequent_indent=indent) for line in str(t).splitlines())

def md_table(df):
    cols = list(df.columns)
    print("| " + " | ".join(cols) + " |")
    print("|" + "---|" * len(cols))
    for _, row in df.iterrows():
        print("| " + " | ".join(str(v) for v in row) + " |")

# ======================= REPORT SECTION 2: rank shifts + Answer 1 vs Answer 2 =======================
print("=" * 100)
print("REPORT SECTION 2 - QUALITATIVE & RANK SHIFT ANALYSIS")
print("=" * 100)
for n, r in enumerate(selected, 1):
    q = r["query"]
    base = baseline_retrieve(q); rr = rerank(q, base)
    b3, r3 = [c for c, _ in base[:TOP_K_FINAL]], [c for c, _ in rr[:TOP_K_FINAL]]
    b_rank = {c: i + 1 for i, (c, _) in enumerate(base)}
    r_rank = {c: i + 1 for i, (c, _) in enumerate(rr)}
    entered = [c for c in r3 if c not in b3]
    dropped = [c for c in b3 if c not in r3]
    a1_declined, a2_declined = r["a1_declined"], r["a2_declined"]

    print(f"\nQUERY {n}  [{r['category']}]")
    print(wrap(q, "  "))
    print(f"  Chunk(s) holding the answer: {r['gold_ids']}  ->  Rank #{r['base_rank']} -> #{r['rerank_rank']}")
    print(f"  Baseline Top 3 : {b3}")
    print(f"  Re-ranked Top 3: {r3}")
    print("  Entered the Top 3 : " + (", ".join(f"chunk {c} (#{b_rank[c]} -> #{r_rank[c]})" for c in entered) or "none"))
    print("  Left the Top 3    : " + (", ".join(f"chunk {c} (#{b_rank[c]} -> #{r_rank[c]})" for c in dropped) or "none"))
    print(f"  Answer 1 said 'context does not contain the answer': {a1_declined} | Answer 2: {a2_declined} | answers identical: {r['ans1'].strip() == r['ans2'].strip()}")
    print(f"  Reference-answer word overlap: Answer 1 {r['overlap1']:.0%} | Answer 2 {r['overlap2']:.0%} | clear answer improvement: {r['answer_improved']}")
    print("  Reference answer:\n" + wrap(r["ref_answer"]))
    print("  Answer 1 (baseline):\n" + wrap(r["ans1"]))
    print("  Answer 2 (re-ranked):\n" + wrap(r["ans2"]))

print("\nOverall effect of re-ranking over ALL evaluated queries (answer chunk inside the baseline Top 10):")
for k, v in effect_counts.items():
    print(f"  {k}: {v}")
print("  (the failure queries above were chosen as illustrative cases from this pool; they are not typical)")

# ======================= REPORT SECTION 3: latency =======================
print("\n" + "=" * 100)
print("REPORT SECTION 3 - LATENCY PROFILING (markdown table, milliseconds, mean of", N_RUNS, "runs)")
print("=" * 100)
md_table(lat.round(2))
avg_row = lat.iloc[-1]
print(f"\nAverage: baseline retrieval {avg_row['baseline_ms']:.1f} ms | re-ranking adds {avg_row['rerank_ms']:.1f} ms | "
      f"Pipeline 2 total {avg_row['pipeline2_total_ms']:.1f} ms  ({avg_row['slowdown_x']:.0f}x the baseline)")
print(f"Measured on: {getattr(cross_encoder.model, 'device', 'cpu')} | chunks indexed: {len(chunks)} | LLM call time excluded (same for both pipelines)")

# ======================= REPORT SECTION 4: numbers behind the scaling question =======================
print("\n" + "=" * 100)
print("REPORT SECTION 4 - SCALING QUESTION (estimate from the measured re-ranking time)")
print("=" * 100)
per_pair_ms = avg_row["rerank_ms"] / TOP_K_RETRIEVE
scale = pd.DataFrame([{
    "collection size (chunks)": n,
    "query-chunk pairs the Cross-Encoder must score": n,
    "estimated time per query": f"{per_pair_ms * n / 1000:,.1f} s" + (f" (~{per_pair_ms * n / 3.6e6:,.1f} h)" if per_pair_ms * n / 1000 > 3600 else ""),
} for n in (TOP_K_RETRIEVE, len(chunks), 10_000, 1_000_000)])
md_table(scale)
print(f"\nBasis: about {per_pair_ms:.0f} ms per (query, chunk) pair, measured from {TOP_K_RETRIEVE} pairs per query on this machine.")
print("Linear extrapolation (batching/GPU would change the constant, not the growth). The bi-encoder embeds all chunks once offline,")
print(f"so a query only needs one embedding + vector search ({avg_row['baseline_ms']:.1f} ms here).")


REPORT SECTION 2 - QUALITATIVE & RANK SHIFT ANALYSIS

QUERY 1  [Keyword Trap / Distractor]
  Under what framework can national governments permit law enforcement to deploy live remote
  biometric scanning in areas open to the public?
  Chunk(s) holding the answer: [440]  ->  Rank #9 -> #3
  Baseline Top 3 : [442, 435, 86]
  Re-ranked Top 3: [223, 95, 440]
  Entered the Top 3 : chunk 223 (#5 -> #1), chunk 95 (#8 -> #2), chunk 440 (#9 -> #3)
  Left the Top 3    : chunk 442 (#1 -> #4), chunk 435 (#2 -> #6), chunk 86 (#3 -> #8)
  Answer 1 said 'context does not contain the answer': False | Answer 2: False | answers identical: False
  Reference-answer word overlap: Answer 1 60% | Answer 2 88% | clear answer improvement: True
  Reference answer:
    A Member State may provide for the authorization of real-time remote biometric identification
    systems in publicly accessible spaces for law enforcement under specific statutory conditions,
    establishing detailed rules in national law for r

## 11. Written analysis (short version; the full report is the PDF)

**Bi-Encoder vs Cross-Encoder.** A Bi-Encoder encodes the query and each chunk separately into fixed vectors; relevance is the cosine between them, so chunk vectors can be precomputed. A Cross-Encoder feeds `[CLS] query [SEP] chunk [SEP]` through one transformer, so every query token attends to every chunk token and a single relevance score comes out. It is more accurate but nothing can be precomputed.

**Rank shifts / answers (numbers from the run above).**
- *Keyword trap* (Art. 5(5), chunk 440): rank 9 -> 3. Answer 1 did not decline but described the conditions of use and missed that a Member State may permit it through detailed national rules; Answer 2 states that.
- *Exception* (Art. 6(3), chunk 446): rank 7 -> 1. Answer 1 declined, Answer 2 gave a valid condition.
- *Negation* (Art. 2(3), chunk 391): rank 7 -> 2, but Answer 1 was already correct (the baseline Top 3 contained chunk 392 and recital chunks stating the same exclusion). This is a rank shift without an answer failure, so it is a weak example (see scan cell 8c / hand-written queries).
- *Other* (Recital 97, chunk 229): rank 8 -> 1. Answer 1 declined, Answer 2 listed all four channels.
- Over all 121 evaluated queries: 24 moved into the Top 3, 2 were pushed out, 92 were in the Top 3 both times, 3 outside both times.
- Overlap with the reference answer is only a rough check, so read the answers yourself.

**Latency.** See Table 3: re-ranking 10 pairs on CPU adds about 1.06 s (about 106 ms per pair), roughly 41x the baseline retrieval time of 26 ms.

**Scaling.** A Cross-Encoder score depends on the (query, chunk) pair, so it needs one forward pass per chunk for every new query and nothing can be cached. At the measured speed this is about 101 s per query for this document and about 29.5 hours for 1 million chunks. The Bi-Encoder embeds the corpus once offline, so a query costs one embedding plus a vector search. Hence: Bi-Encoder for recall over everything, Cross-Encoder for precision over the Top-K only. A chunk missed by the Bi-Encoder (e.g. the emotion-recognition ban in the hand-written query) can never be recovered by the Cross-Encoder.


In [17]:
# Save the final failure cases so the report can be reproduced without re-running the LLM / random sampling
def _clean(o):
    return o.item() if hasattr(o, "item") else str(o)
with open("selected_failure_cases.json", "w") as f:
    json.dump({"model": MODEL, "selected": selected,
               "manual": [m for m in manual_results if m]}, f, indent=2, default=_clean)
print("Saved selected_failure_cases.json")


Saved selected_failure_cases.json
